In [23]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, TimeSeriesSplit
from ElectricSequentialRegressor import ElectricSequentialRegressor
from scipy.stats import randint, uniform
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score, make_scorer
import joblib

<h2>Lectura y Tratamiento de Datos</h2>

In [2]:
X = pd.read_csv('dataset/dataset_in.txt', index_col=0).drop(columns=['P2', 'Q2'])
Y = pd.read_csv('dataset/dataset_out.txt', index_col=0).loc[:, ['PERD', 'QGEN7']]
normalizer_X = MinMaxScaler(feature_range=(0, 1))
X = pd.DataFrame(normalizer_X.fit_transform(X), columns=X.columns, index=X.index)
normalizer_Y = MinMaxScaler(feature_range=(0, 1))
Y = pd.DataFrame(normalizer_Y.fit_transform(Y), columns=Y.columns, index=Y.index)

**Validación por Retención (Tratamiento como Serie Temporal)**

In [3]:
n_samples = X.shape[0]
train_size = int(n_samples * 0.8)
x_train, y_train = X[:train_size], Y[:train_size]
x_test, y_test = X[train_size:], Y[train_size:]

<h2>Búsqueda Aleatoria de Hiperparámetros</h2>

In [4]:
random_model = ElectricSequentialRegressor(RandomForestRegressor(n_jobs=1, random_state=42), RandomForestRegressor(n_jobs=1, random_state=42))
random_grid = {
    'estimator1__n_estimators': randint(30, 101),
    'estimator1__max_depth': randint(10, 51),
    'estimator1__min_samples_split': randint(2, 21),
    'estimator1__max_features': uniform(0.1, 0.9),
    'estimator2__n_estimators': randint(80, 251),
    'estimator2__max_depth': randint(10, 51),
    'estimator2__min_samples_split': randint(2, 21),
    'estimator2__max_features': uniform(0.1, 0.9)
}
def mse_column_score(y_true, y_pred, col_idx):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return mean_squared_error(y_true[:, col_idx], y_pred[:, col_idx])
random_scoring = {
    'mse_global': 'neg_mean_squared_error',
    'mse_perd': make_scorer(mse_column_score, col_idx=0, greater_is_better=False),
    'mse_qgen7': make_scorer(mse_column_score, col_idx=1, greater_is_better=False),
}
random_search = RandomizedSearchCV(random_model, random_grid, n_iter=30,
                                   cv=TimeSeriesSplit(n_splits=3), scoring=random_scoring, refit='mse_global',
                                   n_jobs=-1, verbose=2, random_state=42, return_train_score=True)

In [5]:
random_search.fit(x_train, y_train)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


,estimator,ElectricSeque...dom_state=42))
,param_distributions,"{'estimator1__max_depth': <scipy.stats....0023E61457690>, 'estimator1__max_features': <scipy.stats....0023E5FD03C10>, 'estimator1__min_samples_split': <scipy.stats....0023E623CD3D0>, 'estimator1__n_estimators': <scipy.stats....0023E618C6050>, ...}"
,n_iter,30
,scoring,"{'mse_global': 'neg_mean_squared_error', 'mse_perd': make_scorer(m...t', col_idx=0), 'mse_qgen7': make_scorer(m...t', col_idx=1)}"
,n_jobs,-1
,refit,'mse_global'
,cv,TimeSeriesSpl...est_size=None)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [6]:
best_random_model = random_search.best_estimator_
print('MSE: ' + str(random_search.best_score_))
pd.DataFrame(random_search.cv_results_).loc[:, ['param_estimator1__max_depth', 'param_estimator1__max_features', 'param_estimator1__min_samples_split', 'param_estimator1__n_estimators', 'param_estimator2__max_depth', 'param_estimator2__max_features', 'param_estimator2__min_samples_split', 'param_estimator2__n_estimators', 'mean_test_mse_global', 'mean_test_mse_perd', 'mean_test_mse_qgen7']]

MSE: -0.002169113083781005


,param_estimator1__max_depth,param_estimator1__max_features,param_estimator1__min_samples_split,param_estimator1__n_estimators,param_estimator2__max_depth,param_estimator2__max_features,param_estimator2__min_samples_split,param_estimator2__n_estimators,mean_test_mse_global,mean_test_mse_perd,mean_test_mse_qgen7
0,48,0.816889,16,90,30,0.240417,20,154,-0.002176,-0.000257,-0.004096
1,20,0.879559,5,53,12,0.118526,3,167,-0.002196,-0.000258,-0.004134
2,39,0.291105,2,87,31,0.106360,18,138,-0.002374,-0.000257,-0.004492
3,37,0.976380,16,91,12,0.873946,8,100,-0.002207,-0.000274,-0.004140
4,18,0.158546,5,89,23,0.827558,10,169,-0.003162,-0.000269,-0.006055
5,11,0.715810,16,89,16,0.648997,9,114,-0.002169,-0.000269,-0.004069
6,23,0.451955,19,33,11,0.482640,11,83,-0.002235,-0.000263,-0.004206
7,38,0.972626,13,63,19,0.905345,15,174,-0.002219,-0.000275,-0.004163
8,24,0.179643,9,82,33,0.421078,14,120,-0.003015,-0.000260,-0.005770
9,38,0.821977,2,100,18,0.795020,13,215,-0.002208,-0.000270,-0.004147


In [7]:
random_predictions = best_random_model.predict(x_test)
random_data_score = np.vstack([mean_absolute_error(y_test, random_predictions, multioutput='raw_values').reshape(1,-1),
                              mean_absolute_percentage_error(y_test, random_predictions, multioutput='raw_values').reshape(1,-1),
                              r2_score(y_test, random_predictions, multioutput='raw_values').reshape(1,-1)])
pd.DataFrame(random_data_score, columns=['PERD', 'QGEN7'], index=['MAE', 'MAPE', 'R2'])

,PERD,QGEN7
MAE,0.001261,2.302592e-02
MAPE,0.001958,1.932416e+11
R2,-0.027052,8.849713e-01


<h2>Búsqueda Exhaustiva de Hiperparámetros</h2>

In [17]:
grid_model = ElectricSequentialRegressor(RandomForestRegressor(n_jobs=1, random_state=42), RandomForestRegressor(min_samples_split=18, max_features=0.18, n_jobs=1, random_state=42))
grid = {
    'estimator1__n_estimators': range(80, 90, 5),
    'estimator1__max_depth': range(22, 34, 4),
    'estimator2__n_estimators': range(140, 180, 10),
    'estimator2__max_depth': range(30, 44, 4),
}
def mse_column_score(y_true, y_pred, col_idx):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return mean_squared_error(y_true[:, col_idx], y_pred[:, col_idx])
grid_scoring = {
    'mse_global': 'neg_mean_squared_error',
    'mse_perd': make_scorer(mse_column_score, col_idx=0, greater_is_better=False),
    'mse_qgen7': make_scorer(mse_column_score, col_idx=1, greater_is_better=False),
}
grid_search = GridSearchCV(grid_model, grid, scoring=grid_scoring, cv=TimeSeriesSplit(n_splits=3), n_jobs=-1, verbose=2, return_train_score=True, refit='mse_global')

In [18]:
grid_search.fit(x_train, y_train)

Fitting 3 folds for each of 96 candidates, totalling 288 fits


,estimator,ElectricSeque...dom_state=42))
,param_grid,"{'estimator1__max_depth': range(22, 34, 4), 'estimator1__n_estimators': range(80, 90, 5), 'estimator2__max_depth': range(30, 44, 4), 'estimator2__n_estimators': range(140, 180, 10)}"
,scoring,"{'mse_global': 'neg_mean_squared_error', 'mse_perd': make_scorer(m...t', col_idx=0), 'mse_qgen7': make_scorer(m...t', col_idx=1)}"
,n_jobs,-1
,refit,'mse_global'
,cv,TimeSeriesSpl...est_size=None)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,n_estimators,85


In [20]:
best_grid_model = grid_search.best_estimator_
print('MSE: ' + str(grid_search.best_score_))
pd.DataFrame(grid_search.cv_results_).loc[:, ['param_estimator1__max_depth', 'param_estimator1__n_estimators', 'param_estimator2__max_depth', 'param_estimator2__n_estimators', 'mean_test_mse_global', 'mean_test_mse_perd', 'mean_test_mse_qgen7']]

MSE: -0.0022180536560117065


,param_estimator1__max_depth,param_estimator1__n_estimators,param_estimator2__max_depth,param_estimator2__n_estimators,mean_test_mse_global,mean_test_mse_perd,mean_test_mse_qgen7
0,22,80,30,140,-0.002219,-0.000257,-0.004182
1,22,80,30,150,-0.002219,-0.000256,-0.004182
2,22,80,30,160,-0.002219,-0.000256,-0.004182
3,22,80,30,170,-0.002219,-0.000256,-0.004182
4,22,80,34,140,-0.002219,-0.000256,-0.004182
...,...,...,...,...,...,...,...
91,30,85,38,170,-0.002224,-0.000256,-0.004193
92,30,85,42,140,-0.002224,-0.000256,-0.004193
93,30,85,42,150,-0.002224,-0.000256,-0.004193
94,30,85,42,160,-0.002224,-0.000256,-0.004193


In [21]:
grid_predictions = best_grid_model.predict(x_test)
grid_data_score = np.vstack([mean_absolute_error(y_test, grid_predictions, multioutput='raw_values').reshape(1,-1),
                              mean_absolute_percentage_error(y_test, grid_predictions, multioutput='raw_values').reshape(1,-1),
                              r2_score(y_test, grid_predictions, multioutput='raw_values').reshape(1,-1)])
pd.DataFrame(grid_data_score, columns=['PERD', 'QGEN7'], index=['MAE', 'MAPE', 'R2'])

,PERD,QGEN7
MAE,0.001192,2.332025e-02
MAPE,0.001901,1.958715e+11
R2,0.014007,8.824678e-01


In [24]:
joblib.dump(best_grid_model, 'modelos/ESR_RandomForest.pkl')

['modelos/ESR_RandomForest.pkl']